In [1]:
import pandas as pd
import numpy as np

columns = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3']

for i in range(1, 22):
    columns.append(f'sensor_{i}')

df_train = pd.read_csv('Data/train_FD004.txt', sep='\s+', header=None, names=columns)
sensors_to_drop = ['sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19'] #Dropping sensors with near-zero variance across all conditions as they record flight state, not component wear
df_train = df_train.drop(columns=sensors_to_drop)

df_train.head()

,unit_number,time_in_cycles,setting_1,setting_2,setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
0,1,1,42.0049,0.8400,100.0,549.68,1343.43,1112.93,137.36,2211.86,8311.32,41.69,129.78,2387.99,8074.83,9.3335,330,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,606.07,1477.61,1237.50,332.10,2323.66,8713.60,43.94,312.59,2387.73,8046.13,9.1913,361,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,548.95,1343.12,1117.05,138.18,2211.92,8306.69,41.66,129.62,2387.97,8066.62,9.4007,329,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,548.70,1341.24,1118.03,137.98,2211.88,8312.35,41.68,129.80,2388.02,8076.05,9.3369,328,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,536.10,1255.23,1033.59,174.82,1915.22,7994.94,36.48,164.11,2028.08,7865.80,10.8366,305,14.03,8.6754


In [2]:
rul_calculation = df_train.groupby('unit_number')['time_in_cycles'].max().reset_index()
rul_calculation.columns = ['unit_number', 'max_cycle']
df_train = pd.merge(df_train, rul_calculation, on='unit_number', how='left')

df_train['RUL'] = df_train['max_cycle'] - df_train['time_in_cycles'] #Calculating the true RUL (Remaining Useful Life) for the training set

df_train = df_train.drop('max_cycle', axis=1)

max_rul = 125 #Clipping true labels to match the 125-cycle training ceiling as evaluation must live on the same scale as training
df_train['RUL'] = df_train['RUL'].clip(upper=max_rul)

df_train.head()

,unit_number,time_in_cycles,setting_1,setting_2,setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,549.68,1343.43,1112.93,137.36,2211.86,8311.32,41.69,129.78,2387.99,8074.83,9.3335,330,10.62,6.3670,125
1,1,2,20.0020,0.7002,100.0,606.07,1477.61,1237.50,332.10,2323.66,8713.60,43.94,312.59,2387.73,8046.13,9.1913,361,24.37,14.6552,125
2,1,3,42.0038,0.8409,100.0,548.95,1343.12,1117.05,138.18,2211.92,8306.69,41.66,129.62,2387.97,8066.62,9.4007,329,10.48,6.4213,125
3,1,4,42.0000,0.8400,100.0,548.70,1341.24,1118.03,137.98,2211.88,8312.35,41.68,129.80,2388.02,8076.05,9.3369,328,10.54,6.4176,125
4,1,5,25.0063,0.6207,60.0,536.10,1255.23,1033.59,174.82,1915.22,7994.94,36.48,164.11,2028.08,7865.80,10.8366,305,14.03,8.6754,125


In [3]:
df_test = pd.read_csv('Data/test_FD004.txt', sep='\s+', header=None, names=columns)
df_test = df_test.drop(columns=sensors_to_drop)

true_rul = pd.read_csv('Data/RUL_FD004.txt', sep='\s+', header=None, names=['RUL'])
max_rul = 125 #Limiting final predictions to keep evaluation fair
true_rul['RUL'] = true_rul['RUL'].clip(upper=max_rul)

In [4]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

columns_to_normalize = df_train.columns.difference(['unit_number', 'time_in_cycles', 'RUL', 'op_condition'])

flight_condition_clusterer = KMeans(n_clusters=6, random_state=42, n_init=10) #Using KMeans to automatically sort the data into the 6 distinct flight conditions (altitudes/speeds)
df_train['op_condition'] = flight_condition_clusterer.fit_predict(df_train[['setting_1', 'setting_2', 'setting_3']])
df_test['op_condition']  = flight_condition_clusterer.predict(df_test[['setting_1', 'setting_2', 'setting_3']])

df_train_scaled = df_train.copy()
df_test_scaled  = df_test.copy()
df_train_scaled[columns_to_normalize] = df_train_scaled[columns_to_normalize].astype(float)
df_test_scaled[columns_to_normalize]  = df_test_scaled[columns_to_normalize].astype(float)

condition_scalers = {} #Creating a separate scaler for EACH flight condition so we don't mix high-altitude data with low-altitude data

for condition in sorted(df_train['op_condition'].unique()):
    scaler = MinMaxScaler()
    train_mask = df_train['op_condition'] == condition
    test_mask  = df_test['op_condition']  == condition

    df_train_scaled.loc[train_mask, columns_to_normalize] = scaler.fit_transform(df_train.loc[train_mask, columns_to_normalize]) #Scaling the training data strictly within its own flight regime
    
    if test_mask.any(): #Applying same scale to test data
        df_test_scaled.loc[test_mask, columns_to_normalize] = scaler.transform(df_test.loc[test_mask, columns_to_normalize])
        
    condition_scalers[condition] = scaler

print("Normalization complete")
df_train_scaled.head()

Normalization complete


,unit_number,time_in_cycles,setting_1,setting_2,setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL,op_condition
0,1,1,0.69,0.00,0.0,0.500000,0.255293,0.278866,0.152564,0.283276,...,0.261905,0.206171,0.307937,0.175935,0.602638,0.363636,0.460000,0.473138,125,0
1,1,2,0.40,0.10,0.0,0.113433,0.335117,0.217594,0.156289,0.114650,...,0.241935,0.114089,0.059603,0.115529,0.577645,0.090909,0.297710,0.380397,125,3
2,1,3,0.58,0.45,0.0,0.287791,0.249042,0.349656,0.257692,0.303754,...,0.244048,0.183731,0.301587,0.125431,0.741138,0.272727,0.320000,0.564017,125,0
3,1,4,0.20,0.00,0.0,0.215116,0.211131,0.366495,0.232051,0.290102,...,0.255952,0.208976,0.317460,0.183440,0.609646,0.181818,0.380000,0.557824,125,0
4,1,5,0.83,0.35,0.0,0.220641,0.320654,0.181548,0.237716,0.204918,...,0.283871,0.230321,0.199219,0.171331,0.507187,0.333333,0.222222,0.625683,125,2


In [5]:
sequence_length = 30

X_train_sequences = []
y_train_targets = []
engine_id_per_sequence = [] #Tracking the exact engine for strict isolation

for engine_id in df_train_scaled['unit_number'].unique(): #Looping through each engine individually so we don't accidentally mix the end of engine 1 with the start of engine 2 in the same window
    engine_data = df_train_scaled[df_train_scaled['unit_number'] == engine_id]
    sensor_values = engine_data[columns_to_normalize].values
    rul_values = engine_data['RUL'].values

    for i in range(len(engine_data) - sequence_length): #Sliding the 30-cycle window across the engine's timeline
        X_train_sequences.append(sensor_values[i : i + sequence_length])
        y_train_targets.append(rul_values[i + sequence_length])
        engine_id_per_sequence.append(engine_id) #Tagging the window with its engine ID

X_train = np.array(X_train_sequences)
y_train = np.array(y_train_targets)
engine_ids = np.array(engine_id_per_sequence)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Engine IDs shape: {engine_ids.shape}")

X_train shape: (53779, 30, 17)
y_train shape: (53779,)
Engine IDs shape: (53779,)


In [6]:
np.save('Data/X_train_processed.npy', X_train)
np.save('Data/y_train_processed.npy', y_train)
np.save('Data/engine_ids_train.npy', engine_ids)

In [7]:
X_test_sequences = []

for engine_id in df_test_scaled['unit_number'].unique(): #Extracting exactly one 30-cycle window per engine for grading
    engine_data = df_test_scaled[df_test_scaled['unit_number'] == engine_id]
    sensor_values = engine_data[columns_to_normalize].values

    current_sequence = sensor_values[-sequence_length:]

    if len(current_sequence) < sequence_length: #Zero-padding engines with fewer than 30 recorded cycles so every sequence has the exact same (30, 17) geometry going into the model
        pad_length = sequence_length - len(current_sequence)
        current_sequence = np.pad(current_sequence, pad_width=((pad_length, 0), (0, 0)), mode='constant')

    X_test_sequences.append(current_sequence)

X_test = np.array(X_test_sequences)
y_test = true_rul['RUL'].values

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_test shape: (248, 30, 17)
y_test shape: (248,)


In [8]:
np.save('Data/X_test_processed.npy', X_test)
np.save('Data/y_test_processed.npy', y_test)